In [0]:
%sql
-- File: ddl/create_silver_vehicles.sql
-- Silver layer: cleaned vehicle registry with SCD Type 2 historization.
-- Source: dbr_dev.live_transit_monitor.bronze_vehicles

CREATE TABLE IF NOT EXISTS dbr_dev.live_transit_monitor.silver_vehicles_scd (
    -- business key
    vehicleCode             STRING    NOT NULL COMMENT 'Vehicle code; business key, joins to the GPS stream',

    -- identity attributes
    transportationType      STRING             COMMENT 'Tramwaj / Autobus',
    brand                   STRING             COMMENT 'Manufacturer',
    model                   STRING             COMMENT 'Model designation',
    productionYear          INT                COMMENT 'Year the vehicle was built',
    carrier                 STRING             COMMENT 'Operator (renamed from source typo "carrirer")',
    driveType               STRING             COMMENT 'Drive type, e.g. elektryczny',

    -- capacity and physical attributes
    seats                   INT                COMMENT 'Seated capacity',
    standingPlaces          INT                COMMENT 'Standing capacity',
    length                  DECIMAL(5,2)       COMMENT 'Vehicle length in metres',
    passengersDoors         INT                COMMENT 'Number of passenger doors',
    vehicleCharacteristics  STRING             COMMENT 'Standardowy / Przegubowy / ...',
    floorHeight             STRING             COMMENT 'Floor height category (accessibility)',
    bidirectional           BOOLEAN            COMMENT 'Can operate in both directions',
    historicVehicle         BOOLEAN            COMMENT 'Heritage vehicle',

    -- accessibility and amenities
    wheelchairsRamp         BOOLEAN            COMMENT 'Wheelchair ramp fitted',
    kneelingMechanism       BOOLEAN            COMMENT 'Kneeling suspension fitted',
    airConditioning         BOOLEAN            COMMENT 'Air conditioning fitted',
    usb                     BOOLEAN            COMMENT 'USB charging available',
    bikeHolders             INT                COMMENT 'Number of bicycle holders',
    voiceAnnouncements      BOOLEAN            COMMENT 'Voice announcements fitted',
    monitoring              BOOLEAN            COMMENT 'External monitoring fitted',
    internalMonitor         BOOLEAN            COMMENT 'Internal passenger display fitted',
    aed                     BOOLEAN            COMMENT 'Defibrillator on board',
    ticketMachine           BOOLEAN            COMMENT 'Ticket machine on board',

    -- change detection
    attributes_hash         STRING    NOT NULL COMMENT 'SHA-256 over tracked attributes, used to detect changes',

    -- SCD Type 2 historization
    valid_from              TIMESTAMP NOT NULL COMMENT 'Start of validity for this version',
    valid_to                TIMESTAMP          COMMENT 'End of validity; NULL means currently valid',
    is_current              BOOLEAN   NOT NULL COMMENT 'TRUE for the active version of the vehicle',

    -- lineage
    source_file             STRING             COMMENT 'Source file the record came from',
    silver_ingestion_ts     TIMESTAMP NOT NULL COMMENT 'When this version was written to silver'
)
USING DELTA
COMMENT 'Vehicle registry with SCD Type 2 history. One row per version of a vehicle.';

In [0]:
%sql
-- File: ddl/create_silver_vehicles_quarantine.sql
-- Rejected vehicle records, captured instead of silently dropped.

CREATE TABLE IF NOT EXISTS dbr_dev.live_transit_monitor.silver_vehicles_quarantine_scd (
    vehicleCode         STRING             COMMENT 'Business key, may be NULL if that is why the row failed',
    raw_record          STRING             COMMENT 'Full source record as JSON, for investigation',
    rejection_reasons   ARRAY<STRING>      COMMENT 'Which quality rules the record violated',
    source_file         STRING             COMMENT 'Source file the record came from',
    quarantined_at      TIMESTAMP NOT NULL COMMENT 'When the record was rejected'
)
USING DELTA
COMMENT 'Quarantine for vehicle records failing silver data-quality rules.';